# Week 1 Práctica

## Tutor que responde a preguntas técnicas sobre Python usando dos modelos de IA.

**Modelos implementados:**
- Modelo frontier en la nube (gpt-4o-mini)
- Modelo local (Llama 3.2 via Ollama)

 *La respuesta de GPT se muestra en streaming, palabra a palabra.*

**Modelo single or multi-prompt**
- Modo de funcionamiento por defecto: Single prompt
- Usuario puede elegir cambiar a modo multi-prompt

## 1. Imports
Librerías necesarias para el funcionamiento del notebook.
Incluye el cliente de OpenAI, Ollama, y utilidades para mostrar Markdown en Jupyter.

In [ ]:
# imports

from dotenv import load_dotenv                                     # Carga variables de entorno desde el fichero .env
from IPython.display import Markdown, display, update_display      # Muestra contenido Markdown en Jupyter
from openai import OpenAI                                          # Cliente oficial de OpenAI para llamadas a la API
import ollama                                                      # Cliente oficial de Ollama para modelos locales
import ipywidgets as widgets                                       # Entrada de datos
import os                                                          # Librería estándar de Python para interactuar con el SO
import random                                                      # Librería estándar de Python para generación de valores aleatorios — usada para seleccionar los ejemplos few-shot
import requests                                                    # Librería HTTP de Python — permite hacer llamadas REST a servicios externos como la API de Ollama

## 2. Constantes
Define los nombres de los modelos que vamos a usar.
Centralizar estos valores facilita cambiarlos en un único punto si es necesario.

In [ ]:
# Constantes

# Modelos seleccionables
MODEL_GPT  = 'gpt-4o-mini'   
MODEL_LLAMA = 'llama3.2'     

# Control de respuesta
MAX_TOKENS  = 1200            # Límite máximo de tokens en la respuesta. Riesgo: Respuesta incompleta. Corta respuesta al alcanzar el número de tokens.
MAX_WORDS   = 300             # Límite orientativo de palabras indicado en el system prompt

# Parametro multi-prompt
# default
NUM_EXAMPLES = 2              # Definimos el número de ejemplos en modo multi-prompt. 

# Parámetros Ollama
OLLAMA_STREAM = False                                    # Desactiva el streaming — Ollama devuelve la respuesta completa de una vez
OLLAMA_API    = "http://localhost:11434/api/chat"        # Endpoint REST de Ollama — cambia localhost por otra IP si el servicio está en remoto
HEADERS       = {"Content-Type": "application/json"}     # Indica al servidor que enviamos y esperamos datos en formato JSON

## 3. Selección del modelo de IA

In [ ]:
# Botones por modelo

boton_gpt = widgets.Button(
    description='A — GPT-4o-mini (OpenAI)',
    button_style='primary',                     # Color del botón
    layout=widgets.Layout(width='250px')
)

boton_llama = widgets.Button(
    description='B — Llama 3.2 (Ollama)',
    button_style='success',
    layout=widgets.Layout(width='250px')
)

# Crea el área de salida dentro del notebook donde mostrará los botones 
salida = widgets.Output()                      

 # Todo lo que esta dentro de este bloque aparece en el área de salida
with salida:                                    
    print("Selecciona el modelo de IA: 'A' o 'B'")
    print("")

def elegir_gpt(b):
    global MODEL
    MODEL = MODEL_GPT
    with salida:
        salida.clear_output()
        print(f"✓ Modelo seleccionado: {MODEL}")

def elegir_llama(b):
    global MODEL
    MODEL = MODEL_LLAMA
    with salida:
        salida.clear_output()
        print(f"✓ Modelo seleccionado: {MODEL}")

boton_gpt.on_click(elegir_gpt)
boton_llama.on_click(elegir_llama)

display(widgets.HBox([boton_gpt, boton_llama]), salida)

## 4. Configuración del entorno
Carga las variables del fichero .env, incluyendo la clave de API de OpenAI.
Inicializa el cliente OpenAI listo para hacer llamadas.

In [ ]:
# Configurar entorno

load_dotenv()                           # Carga las variables del fichero .env del directorio raiz desde donde se inició Jupyter Lab
api_key = os.getenv('OPENAI_API_KEY')   # Obtiene la clave de API de OpenAI

# Inicializa el cliente según el modelo seleccionado
if MODEL == MODEL_GPT:
    if not api_key:
        print("⚠️  No se encontró la clave de API de OpenAI en el fichero .env")
    elif not api_key[:8]=='sk-proj-':
        print("⚠️  La clave no tiene un formato válido — debe empezar por sk-")
    elif api_key.strip() != api_key:
        print("⚠️  La clave tiene espacios al principio o al final — corrígelo en el .env")
    else:
        openai = OpenAI()
        print("✓ Cliente OpenAI inicializado correctamente")
else:
    print("✓ Usando Ollama local — no requiere API key")

## 5. La pregunta
Define la pregunta técnica que el tutor va a responder.
Puedes modificar esta celda para preguntar sobre cualquier concepto Python.

In [ ]:
# Registrar la pregunta

# Límite máximo de caracteres para la pregunta
MAX_CHARS = 1000

# Título
titulo = widgets.HTML("<h3>🎓 Tutor técnico de Python</h3><p>Escribe tu pregunta (máx. 1000 caracteres). Para salir escribe <b>SALIR</b>.</p>")

# Área de texto
texto = widgets.Textarea(
    placeholder='Escribe aquí tu pregunta...',
    layout=widgets.Layout(width='600px', height='120px')
)

# Contador de caracteres
contador = widgets.HTML("<small>0 / 1000 caracteres</small>")

# Botones
boton_enviar = widgets.Button(
    description='Enviar al tutor',
    layout=widgets.Layout(width='160px')
)
boton_enviar.style.button_color = '#2E7D32'

boton_borrar = widgets.Button(
    description='Borrar',
    layout=widgets.Layout(width='160px')
)
boton_borrar.style.button_color = '#B71C1C'

# Área de mensajes
salida = widgets.Output()

# Actualizar contador al escribir
def actualizar_contador(change):
    n = len(change['new'])
    color = 'red' if n > MAX_CHARS else 'gray'
    contador.value = f"<small style='color:{color}'>{n} / {MAX_CHARS} caracteres</small>"

texto.observe(actualizar_contador, names='value')

# Acción botón Enviar
def enviar(b):
    global question
    contenido = texto.value.strip()
    with salida:
        salida.clear_output()
        if contenido.upper() == 'SALIR':
            print("👋 Saliendo del tutor. ¡Hasta pronto!")
            boton_enviar.disabled = True
            boton_borrar.disabled = True
        elif not contenido:
            print("⚠️  Estamos esperando tu pregunta. Por favor escribe algo.")
        elif len(contenido) > MAX_CHARS:
            print(f"⚠️  La pregunta supera los {MAX_CHARS} caracteres. Por favor acórtala.")
        else:
            question = contenido
            print(f"✓ Pregunta registrada. Puedes ejecutar la siguiente celda.")

# Acción botón Borrar
def borrar(b):
    texto.value = ''
    with salida:
        salida.clear_output()
        print("🗑️  Pregunta borrada. Escribe una nueva.")

boton_enviar.on_click(enviar)
boton_borrar.on_click(borrar)

# Mostrar todo
display(
    titulo,
    texto,
    contador,
    widgets.HBox([boton_enviar, boton_borrar]),
    salida
)

## 6. System Prompt
Define el rol y comportamiento del modelo de IA.
Indica al modelo que actúe como tutor técnico experto en Python.

In [ ]:
# El prompt de sistema

system_prompt = """
Eres un tutor técnico experto en Python, ingeniería de software y LLMs.
Responde de forma detallada y didáctica, usando ejemplos prácticos de código.
Responde siempre en español y en formato Markdown.
Sé conciso — limita tu respuesta a un máximo de {MAX_WORDS} palabras.
"""

## 7. User Prompt
Construye el mensaje con la pregunta que se envía al modelo.
Combina un texto introductorio fijo con la pregunta definida en el paso anterior.

In [ ]:
# El prompt de usuario

user_prompt = "Por favor, da una explicación detallada de la siguiente pregunta: " + question


## 8. Mensajes
Empaqueta los prompts en el formato estándar que esperan todos los LLMs.

*<b> Modo single-prompt <b/>* 
- Modo por defecto.
- Mensaje con lista con dos elementos: rol system y rol user.

*<b> Modo multi-prompt <b/>*
- Hasta 5 ejemplos predefinidos.
- Habilitado parámetro como constante para poder modificarse por el usuario.
- Valor 2 por defecto para modo mp.
- Lista de ejemplos a empaquetar en el mensaje mediante selección aleatoria.
- Aplico modo aleatorio para enriquecer entradas y utilizar el catágolo de ejemplos codificados

*Mejora para multi-prompt: Capturar los ejemplos mediante una ventana de entrada de datos por parte del usuario*


In [ ]:
# Ejemplos predefinidos para utilizar en modo multi-prompt

EXAMPLES = [
    {
        "user": "¿Qué es una variable en Python?",
        "assistant": """## Variable en Python
Una variable es un contenedor que almacena un valor en memoria.

**Ejemplo:**
```python
nombre = "Chema"   # string
edad = 50          # entero
precio = 9.99      # decimal
```
A diferencia de Java, en Python no se declara el tipo — se infiere automáticamente."""
    },
    {
        "user": "¿Qué hace el operador ** en Python?",
        "assistant": """## Operador ** en Python
El operador `**` calcula la potencia de un número.

**Ejemplo:**
```python
resultado = 2 ** 8   # 2 elevado a 8 = 256
raiz = 9 ** 0.5      # raíz cuadrada de 9 = 3.0
```
Es equivalente a `Math.pow(2, 8)` en Java."""
    },
    {
        "user": "¿Qué es una lista en Python?",
        "assistant": """## Lista en Python
Una lista es una colección ordenada y mutable de elementos.

**Ejemplo:**
```python
frutas = ["manzana", "pera", "uva"]
frutas.append("kiwi")   # añade al final
print(frutas[0])        # "manzana"
```
Equivale a un `ArrayList` en Java."""
    },
    {
        "user": "¿Para qué sirve enumerate()?",
        "assistant": """## enumerate() en Python
Devuelve el índice y el valor en cada iteración de un bucle.

**Ejemplo:**
```python
colores = ["rojo", "verde", "azul"]
for i, color in enumerate(colores):
    print(i, color)   # 0 rojo, 1 verde, 2 azul
```
Evita usar un contador manual como en Java."""
    },
    {
        "user": "¿Qué es un diccionario en Python?",
        "assistant": """## Diccionario en Python
Colección de pares clave-valor, mutable y sin orden garantizado.

**Ejemplo:**
```python
persona = {"nombre": "Chema", "edad": 50}
print(persona["nombre"])   # "Chema"
persona["ciudad"] = "Madrid"
```
Equivale a un `HashMap` en Java."""
    },
]

In [ ]:
# Selector de modo de prompt - 'single' or 'multi'

boton_single = widgets.Button(
    description='Single Prompt',
    button_style='warning',
    layout=widgets.Layout(width='200px')
)

boton_multi = widgets.Button(
    description='Multi Prompt',
    button_style='info',
    layout=widgets.Layout(width='200px')
)

salida_modo = widgets.Output()

with salida_modo:
    print("Selecciona el modo de prompt: 'Single' o 'Multi'")
    print(f"  · Single — solo tu pregunta")
    print(f"  · Multi  — incluye {NUM_EXAMPLES} ejemplos previos (few-shot)")

def aplicar_seleccion(seleccionado, descartado):
    seleccionado.style.button_color = '#00C853'  # verde
    descartado.style.button_color   = '#9E9E9E'  # gris
    seleccionado.disabled = True
    descartado.disabled   = True

def elegir_single(b):
    global PROMPT_MODE
    PROMPT_MODE = "single"
    aplicar_seleccion(boton_single, boton_multi)
    with salida_modo:
        salida_modo.clear_output()
        print(f"✓ Modo seleccionado: Single Prompt")

def elegir_multi(b):
    global PROMPT_MODE
    PROMPT_MODE = "multi"
    aplicar_seleccion(boton_multi, boton_single)
    with salida_modo:
        salida_modo.clear_output()
        print(f"✓ Modo seleccionado: Multi Prompt — se añadirán {NUM_EXAMPLES} ejemplos")

boton_single.on_click(elegir_single)
boton_multi.on_click(elegir_multi)

display(widgets.HBox([boton_single, boton_multi]), salida_modo)

In [ ]:
# Los mensajes en formato lista

if PROMPT_MODE == "single":
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_prompt}
    ]

else:  # multi
    ejemplos_seleccionados = random.sample(EXAMPLES, NUM_EXAMPLES)

    few_shot = []
    for ejemplo in ejemplos_seleccionados:
        few_shot.append({"role": "user",      "content": ejemplo["user"]})
        few_shot.append({"role": "assistant", "content": ejemplo["assistant"]})

    messages = (
        [{"role": "system", "content": system_prompt}]
        + few_shot
        + [{"role": "user", "content": user_prompt}]
    )


## 9. Respuesta GPT-4o-mini con streaming
Llama a la API de OpenAI y muestra la respuesta progresivamente en Markdown.
El streaming mejora la experiencia — el usuario ve la respuesta mientras se genera.

In [ ]:
# Llamada a gpt-4o-mini con streaming

if MODEL == MODEL_GPT:
    stream = openai.chat.completions.create(
        model=MODEL_GPT,
        messages=messages,
        stream=True,        # activa el modo streaming
        max_tokens=MAX_TOKENS
    )

    response = ""
    display_handle = display(Markdown(""), display_id=True)

    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        response = response.replace("```", "").replace("markdown", "")
        update_display(Markdown(response), display_id=display_handle.display_id)



## 10. Respuesta Llama 3.2 (Ollama local)
Llama al modelo local usando la librería ollama.
La respuesta se recibe completa y se muestra formateada en Markdown.

In [ ]:
# Llamada a llama 3.2 via Ollama 

if MODEL == MODEL_LLAMA:
    payload = {
        "model"    : MODEL_LLAMA,
        "messages" : messages,
        "stream"   : OLLAMA_STREAM
    }

    response      = requests.post(OLLAMA_API, json=payload, headers=HEADERS)
    response_json = response.json()
    reply         = response_json.get('message', {}).get('content', 'No content received')
    display(Markdown(reply))